# 유방 초음파 종양 분할 (BUSI) 불균형 세그멘테이션 실험

---

## 1. 태스크 및 도메인
- **도메인**: 유방 초음파 종양 (Breast Ultrasound Tumor) 세그멘테이션
- **모달리티**: Ultrasound (초음파 그레이스케일)
- **태스크**: Binary segmentation — 배경(0) / 종양(1)
- **핵심 도전**: 종양 크기 편차 극심(소형~대형), speckle 노이즈, benign/malignant 형태 차이, 불명확한 병변 경계

## 2. 모델
- **아키텍처**: U-Net (ResNet34 백본)
- **사전학습**: ImageNet pretrained
- **선택 이유**: Binary segmentation 표준 베이스라인, 다도메인 비교 일관성
- **출력**: 1채널 sigmoid → `to_2ch_logits()` 변환 후 손실 함수 적용
- **입력**: Grayscale → 3채널 복제 (ImageNet 인코더 3ch 맞춤)

## 3. 데이터셋
- **이름**: BUSI (Breast Ultrasound Images Dataset)
- **규모**: 647장 — benign 437장 + malignant 210장 (normal 133장 제외)
  - Train 80% / Val 10% / Test 10% (이미지 단위 랜덤 분할)
- **입력 해상도**: 256×256 (리사이즈)
- **클래스 불균형**: BG:Tumor = **~3~10:1** (종양 크기에 따라 편차 큼)
- **공식 분할**: 없음 → 이미지 단위 8:1:1 랜덤 분할 (random_state=42)

## 4. 데이터 준비 (협업자용)
> Cell 0 실행 시 kagglehub로 자동 다운로드. 별도 준비 불필요.

**취득 방법 (자동)**:
```python
kagglehub.dataset_download("aryashah2k/breast-ultrasound-images-dataset")
```
공식 논문: Al-Dhabyani W, et al. "Dataset of breast ultrasound images." Data in Brief, 2020.

**폴더 구조** (다운로드 후):
```
Dataset_BUSI_with_GT/
  benign/
    benign (1).png
    benign (1)_mask.png   ← 흰색=종양, 검정=배경
    ...
  malignant/
    malignant (1).png
    malignant (1)_mask.png
    ...
  normal/                 ← 실험에서 제외 (종양 없음)
```

## 5. 전처리 및 도메인 특이점
- Grayscale → 3채널 복제 (ImageNet 인코더 3ch 채널 수 맞춤)
- 마스크 이진화 (임계값 128): pixel > 128 → 종양(1), 나머지 → 배경(0)
- 다중 마스크 파일 존재 시 (`_mask_1.png`, `_mask_2.png`) 합산(union) 처리
- Normal 클래스(133장) 제외 — 마스크가 없거나 전체 배경
- ImageNet 정규화 (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

## 6. 실험 손실 함수 및 Optuna 탐색 범위
| 손실 함수 | 탐색 파라미터 | 탐색 범위 | Trials |
|-----------|-------------|----------|--------|
| `ce_dice` | — | — | — |
| `wce_dice` | — | — | — |
| `lwce_dice` | — | — | — |
| `plwce_dice` | alpha | 2.5 ~ 15.0 | 20 |
| `pwce_dice` | alpha | 0.2 ~ 2.5 | 20 |
| `cb_dice` | — | — | — |
| `plwce_focal_dice` | alpha + gamma | alpha 2.5~15.0, gamma 0.5~5.0 | 40 |

## 7. SoTA 참고 (2026년 3월 기준)
| 방법 | Dice | IoU | 출처 |
|------|------|-----|------|
| TransUNet+Att (2024) | ~87% | ~80% | arXiv |
| U-Net++ baseline | ~79~83% | ~70~75% | 복수 논문 |
| U-Net baseline | ~72~78% | ~65~70% | 복수 논문 |

> 본 연구 목표: U-Net baseline 대비 LWCE/PLWCE 계열 손실 함수의 개선 효과 검증.
> 평가 지표: Dice, Sensitivity, Specificity, AUC
> 결과 저장: `medical_data/results/BUSI_Breast_Ultrasound/`

In [ ]:
# === Cell 0: 환경설정 ===
import os, sys, random, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

import optuna
import kagglehub
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# --- custom_losses 경로 ---
_CL_LOCAL = '/root/imbalanced-data-LWCE/medical_data'
_CL_COLAB = '/tmp/custom_losses'
if os.path.exists(_CL_LOCAL):
    sys.path.insert(0, _CL_LOCAL)
else:
    sys.path.insert(0, _CL_COLAB)
from custom_losses import get_loss_function, calculate_weights

# --- 디바이스 ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# --- 시드 고정 ---
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- Colab 환경 감지 (BUSI는 kagglehub 자동 다운로드 — Drive 마운트 불필요) ---
IS_COLAB = False
try:
    import google.colab
    IS_COLAB = True
    # custom_losses.py Colab 복사
    import shutil as _sh
    _cl_src = '/content/drive/MyDrive/imbalanced-data-LWCE/medical_data/custom_losses.py'
    if not os.path.exists(os.path.join(_CL_LOCAL, 'custom_losses.py')) and os.path.exists(_cl_src):
        os.makedirs(_CL_COLAB, exist_ok=True)
        _sh.copy(_cl_src, _CL_COLAB)
    print('Colab 환경 감지')
except ImportError:
    print('로컬 환경')

# --- 결과 저장 경로 ---
RESULTS_DIR = '/root/imbalanced-data-LWCE/medical_data/results/BUSI_Breast_Ultrasound'
os.makedirs(RESULTS_DIR, exist_ok=True)

# --- 하이퍼파라미터 ---
IMG_SIZE      = 256
BATCH_SIZE    = 16
NUM_WORKERS   = 2
FINAL_EPOCHS  = 100
FINAL_LR      = 1e-4
PROXY_EPOCHS  = 10
PROXY_SUBSET  = 0.15
N_TRIALS      = 20
N_TRIALS_PF   = 40

NUM_CLASSES   = 2
CLASS_NAMES   = ['BG', 'Tumor']

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

print('환경설정 완료')

In [ ]:
# === Cell 1: 데이터 ===

# --- kagglehub 자동 다운로드 ---
print('BUSI 데이터셋 다운로드 중 (최초 1회, 이후 캐시 사용)...')
dataset_path = kagglehub.dataset_download('aryashah2k/breast-ultrasound-images-dataset')
print(f'다운로드 경로: {dataset_path}')

# Dataset_BUSI_with_GT/ 하위 폴더 탐색
BUSI_ROOT = dataset_path
for candidate in [os.path.join(dataset_path, 'Dataset_BUSI_with_GT'),
                  dataset_path]:
    if os.path.isdir(os.path.join(candidate, 'benign')):
        BUSI_ROOT = candidate
        break
print(f'BUSI 루트: {BUSI_ROOT}')

# --- 파일 목록 수집 (benign + malignant, normal 제외) ---
def collect_busi_pairs(root):
    """
    (img_path, mask_path) 쌍 수집.
    다중 마스크(_mask_1, _mask_2)가 있는 경우 union으로 합산.
    """
    pairs = []
    for category in ['benign', 'malignant']:
        cat_dir = os.path.join(root, category)
        if not os.path.isdir(cat_dir):
            continue
        all_files = os.listdir(cat_dir)
        # 이미지 파일만 선별 (마스크 제외)
        img_files = [
            f for f in sorted(all_files)
            if not '_mask' in f
            and f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))
        ]
        for img_fname in img_files:
            stem = os.path.splitext(img_fname)[0]
            img_path = os.path.join(cat_dir, img_fname)
            # 기본 마스크 경로
            mask_candidates = [
                os.path.join(cat_dir, stem + '_mask.png'),
                os.path.join(cat_dir, stem + '_mask.jpg'),
            ]
            mask_path = next((m for m in mask_candidates if os.path.exists(m)), None)
            if mask_path is not None:
                pairs.append((img_path, mask_path, cat_dir, stem))
    return pairs

raw_pairs = collect_busi_pairs(BUSI_ROOT)
print(f'총 이미지-마스크 쌍: {len(raw_pairs)}')

# --- Train / Val / Test 분할 (이미지 단위, 8:1:1) ---
# (img_path, mask_path) 형태로 단순화
simple_pairs = [(p[0], p[1]) for p in raw_pairs]
tr_pairs, tmp_pairs   = train_test_split(simple_pairs, test_size=0.2, random_state=SEED)
val_pairs, test_pairs = train_test_split(tmp_pairs,    test_size=0.5, random_state=SEED)
print(f'Train: {len(tr_pairs)}, Val: {len(val_pairs)}, Test: {len(test_pairs)}')

# --- 다중 마스크 합산 함수 ---
def load_mask_union(img_path, mask_path):
    """
    기본 마스크 + _mask_1, _mask_2 등 추가 마스크가 있으면 union 처리.
    반환: (H, W) uint8 이진 마스크
    """
    cat_dir = os.path.dirname(mask_path)
    stem    = os.path.splitext(os.path.basename(img_path))[0]

    mask_union = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask_union is None:
        h, w = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE).shape[:2]
        return np.zeros((h, w), dtype=np.uint8)

    # 추가 마스크 탐색
    for suffix in ['_mask_1', '_mask_2', '_mask_3']:
        extra = os.path.join(cat_dir, stem + suffix + '.png')
        if os.path.exists(extra):
            extra_mask = cv2.imread(extra, cv2.IMREAD_GRAYSCALE)
            if extra_mask is not None:
                mask_union = np.maximum(mask_union, extra_mask)

    return (mask_union > 128).astype(np.uint8)  # 이진화

# --- Dataset ---
def to_2ch_logits(p):
    """1채널 sigmoid 출력 → 2채널 logit 변환"""
    return torch.cat([-p, p], dim=1)

class BUSIDataset(Dataset):
    def __init__(self, pairs, augment=False):
        self.pairs   = pairs
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        # 이미지: Grayscale → 3채널 복제
        img_gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        img_gray = cv2.resize(img_gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        img_3ch  = np.stack([img_gray, img_gray, img_gray], axis=2)  # (H, W, 3)

        # 마스크
        mask = load_mask_union(img_path, mask_path)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

        # ImageNet 정규화
        img_3ch = (img_3ch - IMAGENET_MEAN) / IMAGENET_STD

        # 증강
        if self.augment:
            if random.random() > 0.5:
                img_3ch = np.fliplr(img_3ch).copy()
                mask    = np.fliplr(mask).copy()
            if random.random() > 0.5:
                img_3ch = np.flipud(img_3ch).copy()
                mask    = np.flipud(mask).copy()
            k = random.randint(0, 3)
            if k > 0:
                img_3ch = np.rot90(img_3ch, k).copy()
                mask    = np.rot90(mask,    k).copy()

        img_t  = torch.from_numpy(img_3ch.transpose(2, 0, 1)).float()  # (3, H, W)
        mask_t = torch.from_numpy(mask.astype(np.int64))                # (H, W)
        return img_t, mask_t

train_ds   = BUSIDataset(tr_pairs,   augment=True)
val_ds     = BUSIDataset(val_pairs,  augment=False)
test_ds    = BUSIDataset(test_pairs, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# --- 클래스 비율 계산 (학습 데이터 기준) ---
class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for _, mask_t in tqdm(train_loader, desc='클래스 비율 계산'):
    class_counts[0] += int((mask_t == 0).sum())
    class_counts[1] += int((mask_t == 1).sum())

total = class_counts.sum()
print('\n클래스 비율:')
for name, count in zip(CLASS_NAMES, class_counts):
    print(f'  {name}: {count:,} ({count / total * 100:.2f}%)')
print(f'  BG:Tumor = {class_counts[0] / class_counts[1]:.1f}:1')

In [ ]:
# === Cell 2: 모델 ===

# --- U-Net (ResNet34) — 1채널 출력 (Binary) ---
def build_model():
    return smp.Unet(
        encoder_name    = 'resnet34',
        encoder_weights = 'imagenet',
        in_channels     = 3,
        classes         = 1,
        activation      = None,
    ).to(device)

# --- 검증 지표: Dice, Sensitivity, Specificity, AUC ---
def compute_val_metrics(model, loader):
    model.eval()
    all_probs = []
    all_preds = []
    all_masks = []

    with torch.no_grad():
        for imgs, masks in loader:
            imgs  = imgs.to(device)
            probs = torch.sigmoid(model(imgs)).squeeze(1).cpu()  # (B, H, W)
            preds = (probs > 0.5).long()
            all_probs.append(probs.numpy().flatten())
            all_preds.append(preds.numpy().flatten())
            all_masks.append(masks.numpy().flatten())

    probs_np = np.concatenate(all_probs)
    preds_np = np.concatenate(all_preds)
    masks_np = np.concatenate(all_masks)

    # Dice
    inter = ((preds_np == 1) & (masks_np == 1)).sum()
    union = (preds_np == 1).sum() + (masks_np == 1).sum()
    dice  = float(2 * inter / (union + 1e-8))

    # Sensitivity (Recall), Specificity
    tp = ((preds_np == 1) & (masks_np == 1)).sum()
    fn = ((preds_np == 0) & (masks_np == 1)).sum()
    tn = ((preds_np == 0) & (masks_np == 0)).sum()
    fp = ((preds_np == 1) & (masks_np == 0)).sum()
    sensitivity  = float(tp / (tp + fn + 1e-8))
    specificity  = float(tn / (tn + fp + 1e-8))

    # AUC
    try:
        auc = float(roc_auc_score(masks_np, probs_np))
    except Exception:
        auc = 0.0

    return {
        'Dice':        dice,
        'Sensitivity': sensitivity,
        'Specificity': specificity,
        'AUC':         auc,
    }

In [ ]:
# === Cell 3: 학습함수 ===

def train_model(loss_name, alpha=1.0, gamma=2.0, epochs=50, lr=1e-4,
                subset_ratio=1.0, tag=''):
    """
    BUSI Binary 세그멘테이션 학습.
    1채널 sigmoid 출력 → to_2ch_logits 변환 후 criterion 적용.
    """
    model     = build_model()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = get_loss_function(loss_name, class_counts=class_counts, alpha=alpha, gamma=gamma)

    # --- 서브셋 로더 (Optuna proxy) ---
    if subset_ratio < 1.0:
        n      = max(1, int(len(train_ds) * subset_ratio))
        sub_ds = torch.utils.data.Subset(train_ds, random.sample(range(len(train_ds)), n))
        loader = DataLoader(sub_ds, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=True)
    else:
        loader = train_loader

    best_dice  = 0.0
    best_state = None
    history    = {'loss': [], 'val_dice': []}
    ckpt_path  = f'/tmp/best_busi_{tag}_{loss_name}.pth'

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for imgs, masks in tqdm(loader,
                                desc=f'[{tag}] {loss_name} Ep{epoch+1:02d}/{epochs}',
                                leave=False):
            imgs  = imgs.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()
            logits_2ch = to_2ch_logits(model(imgs))  # (B, 2, H, W)
            loss = criterion(logits_2ch, masks)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()

        val_metrics = compute_val_metrics(model, val_loader)
        val_dice    = val_metrics['Dice']
        history['loss'].append(epoch_loss / len(loader))
        history['val_dice'].append(val_dice)

        if val_dice > best_dice:
            best_dice  = val_dice
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_state, ckpt_path)

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history, best_dice

In [ ]:
# === Cell 4: Optuna ===
os.environ['TQDM_DISABLE'] = '1'

# --- 탐색 범위 ---
ALPHA_LOW_PLWCE  = 2.5;  ALPHA_HIGH_PLWCE = 15.0
ALPHA_LOW_PWCE   = 0.2;  ALPHA_HIGH_PWCE  = 2.5
GAMMA_LOW        = 0.5;  GAMMA_HIGH       = 5.0

def make_objective(loss_name, alpha_low, alpha_high, gamma_low=None, gamma_high=None):
    def objective(trial):
        alpha = trial.suggest_float('alpha', alpha_low, alpha_high)
        gamma = trial.suggest_float('gamma', gamma_low, gamma_high) if gamma_low is not None else 2.0
        try:
            _, _, dice = train_model(
                loss_name    = loss_name,
                alpha        = alpha,
                gamma        = gamma,
                epochs       = PROXY_EPOCHS,
                subset_ratio = PROXY_SUBSET,
                tag          = f'trial{trial.number}',
            )
            return dice
        except Exception as e:
            print(f'Trial {trial.number} 실패: {e}')
            return 0.0
    return objective

# --- PLWCE alpha 탐색 ---
print('=== PLWCE alpha 탐색 ===')
study_plwce = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)
)
study_plwce.optimize(make_objective('plwce_dice', ALPHA_LOW_PLWCE, ALPHA_HIGH_PLWCE),
                     n_trials=N_TRIALS, show_progress_bar=False)
best_alpha_plwce = study_plwce.best_params['alpha']
print(f'PLWCE best alpha: {best_alpha_plwce:.4f}  Dice: {study_plwce.best_value:.4f}')

# --- PWCE alpha 탐색 ---
print('=== PWCE alpha 탐색 ===')
study_pwce = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)
)
study_pwce.optimize(make_objective('pwce_dice', ALPHA_LOW_PWCE, ALPHA_HIGH_PWCE),
                    n_trials=N_TRIALS, show_progress_bar=False)
best_alpha_pwce = study_pwce.best_params['alpha']
print(f'PWCE best alpha: {best_alpha_pwce:.4f}  Dice: {study_pwce.best_value:.4f}')

# --- PLWCE+Focal alpha+gamma 탐색 ---
print('=== PLWCE+Focal alpha+gamma 탐색 ===')
study_pf = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)
)
study_pf.optimize(
    make_objective('plwce_focal_dice', ALPHA_LOW_PLWCE, ALPHA_HIGH_PLWCE, GAMMA_LOW, GAMMA_HIGH),
    n_trials=N_TRIALS_PF, show_progress_bar=False
)
best_alpha_pf = study_pf.best_params['alpha']
best_gamma_pf = study_pf.best_params['gamma']
print(f'PLWCE+Focal best alpha: {best_alpha_pf:.4f}, gamma: {best_gamma_pf:.4f}  Dice: {study_pf.best_value:.4f}')

os.environ.pop('TQDM_DISABLE', None)

# --- 탐색 시각화 (study 정의 이후) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('BUSI Breast Ultrasound — Optuna Alpha/Gamma 탐색 결과')

for ax, study, name in [
    (axes[0], study_plwce, 'PLWCE'),
    (axes[1], study_pwce,  'PWCE'),
]:
    trials = [t for t in study.trials if t.value is not None]
    xs     = [t.params['alpha'] for t in trials]
    ys     = [t.value for t in trials]
    ax.scatter(xs, ys, alpha=0.6, s=40, color='steelblue')
    bx = study.best_params['alpha']
    by = study.best_value
    ax.axvline(bx, color='red', linestyle='--', linewidth=1.5, label=f'Best alpha={bx:.3f}')
    ax.scatter([bx], [by], color='red', s=100, zorder=5)
    ax.set_xlabel('alpha'); ax.set_ylabel('Val Dice')
    ax.set_title(f'{name} alpha 탐색'); ax.legend(); ax.grid(True)

ax = axes[2]
trials_pf = [t for t in study_pf.trials if t.value is not None]
alphas_pf = [t.params['alpha'] for t in trials_pf]
gammas_pf = [t.params['gamma'] for t in trials_pf]
values_pf = [t.value for t in trials_pf]
sc = ax.scatter(alphas_pf, gammas_pf, c=values_pf, cmap='viridis', alpha=0.7, s=40)
ax.scatter([best_alpha_pf], [best_gamma_pf], color='red', s=150, zorder=5,
           marker='*', label=f'Best α={best_alpha_pf:.3f}, γ={best_gamma_pf:.3f}')
plt.colorbar(sc, ax=ax, label='Val Dice')
ax.set_xlabel('alpha'); ax.set_ylabel('gamma')
ax.set_title('PLWCE+Focal alpha+gamma 탐색'); ax.legend(); ax.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'BUSI_optuna_search.png'), dpi=100)
plt.close()
print(f'탐색 결과 이미지 저장: {RESULTS_DIR}/BUSI_optuna_search.png')

In [ ]:
# === Cell 5: 학습실행 ===

experiments = [
    ('ce_dice',          1.0,              2.0,           'CE+Dice (baseline)'),
    ('wce_dice',         1.0,              2.0,           'WCE+Dice'),
    ('lwce_dice',        1.0,              2.0,           'LWCE+Dice'),
    ('plwce_dice',       best_alpha_plwce, 2.0,           f'PLWCE+Dice (α={best_alpha_plwce:.2f})'),
    ('pwce_dice',        best_alpha_pwce,  2.0,           f'PWCE+Dice (α={best_alpha_pwce:.2f})'),
    ('cb_dice',          1.0,              2.0,           'CB+Dice'),
    ('plwce_focal_dice', best_alpha_pf,    best_gamma_pf, f'PLWCE+Focal+Dice (α={best_alpha_pf:.2f}, γ={best_gamma_pf:.2f})'),
]

all_results = {}
for loss_name, alpha, gamma, label in experiments:
    print(f'\n{"="*60}')
    print(f'학습: {label}')
    print(f'{"="*60}')
    model, history, best_dice = train_model(
        loss_name = loss_name,
        alpha     = alpha,
        gamma     = gamma,
        epochs    = FINAL_EPOCHS,
        lr        = FINAL_LR,
        tag       = 'final',
    )
    all_results[label] = {
        'model':         model,
        'history':       history,
        'best_val_dice': best_dice,
        'loss_name':     loss_name,
        'alpha':         alpha,
        'gamma':         gamma,
    }
    print(f'  Best Val Dice: {best_dice:.4f}')

print('\n모든 학습 완료!')

In [ ]:
# === Cell 6: 평가/저장 ===

# --- Test set 최종 평가 ---
print('=== Test Set 최종 평가 ===')
final_results = {}
for label, v in all_results.items():
    metrics = compute_val_metrics(v['model'], test_loader)
    final_results[label] = {
        'loss_name':     v['loss_name'],
        'alpha':         v['alpha'],
        'gamma':         v['gamma'],
        'best_val_dice': v['best_val_dice'],
        **metrics,
    }
    print(f'{label}: Dice={metrics["Dice"]:.4f}  Sens={metrics["Sensitivity"]:.4f}  '
          f'Spec={metrics["Specificity"]:.4f}  AUC={metrics["AUC"]:.4f}')

# --- 학습 곡선 ---
n_exp  = len(all_results)
n_cols = 4
n_rows = (n_exp + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4))
axes = axes.flatten()
fig.suptitle('BUSI Breast Ultrasound — 학습 곡선 (Loss & Val Dice)')

for i, (label, v) in enumerate(all_results.items()):
    ax  = axes[i]
    ax2 = ax.twinx()
    ep  = range(1, len(v['history']['loss']) + 1)
    ax.plot(ep,  v['history']['loss'],     'b-', alpha=0.7, label='Train Loss')
    ax2.plot(ep, v['history']['val_dice'], 'r-', alpha=0.7, label='Val Dice')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss', color='b')
    ax2.set_ylabel('Val Dice', color='r')
    ax.set_title(label, fontsize=9)
    ax.legend(loc='upper left', fontsize=7)
    ax2.legend(loc='upper right', fontsize=7)
    ax.grid(True, alpha=0.3)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'BUSI_training_curves.png'), dpi=100)
plt.close()
print(f'학습 곡선 저장: {RESULTS_DIR}/BUSI_training_curves.png')

# --- 예측 시각화 ---
best_label = max(final_results, key=lambda k: final_results[k]['Dice'])
best_model = all_results[best_label]['model']
best_model.eval()

sample_imgs, sample_masks = next(iter(test_loader))
with torch.no_grad():
    sample_probs = torch.sigmoid(best_model(sample_imgs.to(device))).squeeze(1).cpu()
    sample_preds = (sample_probs > 0.5).long()

n_show = min(4, len(sample_imgs))
fig, axes = plt.subplots(n_show, 4, figsize=(16, 4 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]
fig.suptitle(f'BUSI 예측 시각화 (Best: {best_label})')

for i in range(n_show):
    img_vis  = sample_imgs[i, 0].numpy()       # 그레이스케일 첫 채널
    gt_np    = sample_masks[i].numpy()
    pred_np  = sample_preds[i].numpy()
    prob_np  = sample_probs[i].numpy()
    axes[i, 0].imshow(img_vis,  cmap='gray');  axes[i, 0].set_title('Input');        axes[i, 0].axis('off')
    axes[i, 1].imshow(gt_np,    cmap='gray');  axes[i, 1].set_title('Ground Truth'); axes[i, 1].axis('off')
    axes[i, 2].imshow(prob_np,  cmap='hot');   axes[i, 2].set_title('Prob Map');     axes[i, 2].axis('off')
    axes[i, 3].imshow(pred_np,  cmap='gray');  axes[i, 3].set_title('Prediction');   axes[i, 3].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'BUSI_prediction_vis.png'), dpi=100)
plt.close()
print(f'예측 시각화 저장: {RESULTS_DIR}/BUSI_prediction_vis.png')

# --- 최종 지표 바차트 ---
labels_plot  = list(final_results.keys())
short_labels = [k.split('(')[0].strip() for k in labels_plot]
dice_v = [final_results[k]['Dice']        for k in labels_plot]
sens_v = [final_results[k]['Sensitivity'] for k in labels_plot]
spec_v = [final_results[k]['Specificity'] for k in labels_plot]
auc_v  = [final_results[k]['AUC']         for k in labels_plot]

x = np.arange(len(labels_plot))
w = 0.2
fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(x - 1.5*w, dice_v, w, label='Dice',        color='steelblue',      alpha=0.85)
ax.bar(x - 0.5*w, sens_v, w, label='Sensitivity', color='tomato',         alpha=0.85)
ax.bar(x + 0.5*w, spec_v, w, label='Specificity', color='mediumseagreen', alpha=0.85)
ax.bar(x + 1.5*w, auc_v,  w, label='AUC',         color='mediumpurple',   alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(short_labels, rotation=20, ha='right')
ax.set_ylabel('Score')
ax.set_title('BUSI Breast Ultrasound — Loss별 최종 성능 비교')
ax.legend(); ax.grid(True, axis='y', alpha=0.3); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'BUSI_final_metrics.png'), dpi=100)
plt.close()
print(f'최종 지표 바차트 저장: {RESULTS_DIR}/BUSI_final_metrics.png')

# --- JSON 저장 ---
save_data = {
    'domain':      'BUSI Breast Ultrasound Tumor Segmentation',
    'model':       'U-Net (ResNet34, ImageNet pretrained)',
    'num_classes': NUM_CLASSES,
    'class_names': CLASS_NAMES,
    'class_counts': {n: int(c) for n, c in zip(CLASS_NAMES, class_counts)},
    'imbalance': {
        'BG:Tumor': round(float(class_counts[0]) / float(class_counts[1]), 1)
    },
    'optuna': {
        'best_alpha_plwce': best_alpha_plwce,
        'best_alpha_pwce':  best_alpha_pwce,
        'best_alpha_pf':    best_alpha_pf,
        'best_gamma_pf':    best_gamma_pf,
    },
    'results': {
        k: {mk: float(mv) if isinstance(mv, (float, np.floating)) else mv
            for mk, mv in v.items()}
        for k, v in final_results.items()
    },
    'best_model': max(final_results, key=lambda k: final_results[k]['Dice']),
}
json_path = os.path.join(RESULTS_DIR, 'BUSI_final_results.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(save_data, f, indent=2, ensure_ascii=False)
print(f'JSON 저장: {json_path}')

# --- Excel 저장 ---
summary_rows = []
for label, v in final_results.items():
    summary_rows.append({
        'Loss Function': label,
        'alpha':         round(v['alpha'], 4),
        'gamma':         round(v['gamma'], 4),
        'Dice':          round(v['Dice'],        4),
        'Sensitivity':   round(v['Sensitivity'], 4),
        'Specificity':   round(v['Specificity'], 4),
        'AUC':           round(v['AUC'],         4),
        'Best_Val_Dice': round(v['best_val_dice'], 4),
    })

history_rows = []
for label, v in all_results.items():
    for ep, (loss_val, dice_val) in enumerate(
            zip(v['history']['loss'], v['history']['val_dice']), 1):
        history_rows.append({
            'Loss Function': label,
            'Epoch':         ep,
            'Train Loss':    round(loss_val, 6),
            'Val Dice':      round(dice_val, 6),
        })

df_summary = pd.DataFrame(summary_rows)
df_history = pd.DataFrame(history_rows)

excel_path = os.path.join(RESULTS_DIR, 'BUSI_final_results.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_summary.to_excel(writer, sheet_name='Summary',          index=False)
    df_history.to_excel(writer, sheet_name='Training_History', index=False)
print(f'Excel 저장: {excel_path}')

# --- 최종 요약 출력 ---
print('\n=== 최종 결과 요약 ===')
print(df_summary.to_string(index=False))
print(f'\n최고 성능 모델: {save_data["best_model"]}')